# RoadVision - Training Session One

Nashat Alfarajat - Laith hamaydeh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
-"""
NEXT time (zip the dataset then put it on drive)
# Much faster to copy one big file than thousands of small images
!cp "/content/drive/MyDrive/dataset.zip" "/content/"
           change the path ^
=========
# Unzip fast on local SSD
!unzip -q /content/dataset.zip -d /content/Final_Ready_Dataset_RV
              path change ^              ^
"""

'\nNEXT time (zip the dataset then put it on drive)\n# Much faster to copy one big file than thousands of small images\n!cp "/content/drive/MyDrive/dataset.zip" "/content/"\n           change the path ^\n=========\n# Unzip fast on local SSD\n!unzip -q /content/dataset.zip -d /content/Final_Ready_Dataset_RV\n              path change ^              ^\n'

In [ ]:
!cp "/content/drive/MyDrive/Final_Ready_Dataset_RV.zip" "/content/"

In [ ]:
# is uploade complete
!ls -lh /content/Final_Ready_Dataset_RV.zip


-rw------- 1 root root 14G May  5 21:12 /content/Final_Ready_Dataset_RV.zip


In [ ]:
!unzip  /content/Final_Ready_Dataset_RV.zip -d /content/Final_Ready_Dataset_RV

Streaming output truncated to the last 5000 lines.
  inflating: /content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train/d1_f9571fb4-img004283_aug_mirror.txt  
  inflating: /content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train/d1_f97e697e-img003700.txt  
  inflating: /content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train/d1_f97e697e-img003700_aug_brightness.txt  
  inflating: /content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train/d1_f97e697e-img003700_aug_contrast.txt  
  inflating: /content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train/d1_f97e697e-img003700_aug_mirror.txt  
  inflating: /content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train/d1_f98688ab-img002459.txt  
  inflating: /content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train/d1_f98688ab-img002459_aug_brightness.txt  
  inflating: /content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train/d1_f98688ab-img002459_aug_contrast.txt  

In [ ]:
!du -sh /content/Final_Ready_Dataset_RV

15G	/content/Final_Ready_Dataset_RV


In [ ]:
# chaneg path on data.yaml file
import yaml

yaml_path = "/content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/data.yaml"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

data["path"] = "/content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV"

with open(yaml_path, "w") as f:
    yaml.dump(data, f)

print("Updated data.yaml ✓")

Updated data.yaml ✓


In [ ]:
import os
images = "/content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/images/train"
labels = "/content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/labels/train"

image_files = {os.path.splitext(f)[0] for f in os.listdir(images)}
label_files = {os.path.splitext(f)[0] for f in os.listdir(labels)}

missing_labels = image_files - label_files
missing_images = label_files - image_files

print("Images:", len(image_files))
print("Labels:", len(label_files))
print("train Images without labels:", len(missing_labels))
print("train Labels without images:", len(missing_images))

Images: 20382
Labels: 20382
train Images without labels: 0
train Labels without images: 0


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.2 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import torch

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
model = YOLO("yolo26s.pt")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

model.train(
    # ── Core ──────────────────────────────────────────────
    data="/content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/data.yaml",
    epochs=50,
    time=None,                  # Max training hours (overrides epochs if set)
    patience=100,               # Early stopping patience (0 to disable)
    imgsz=640,
    batch=32,
    fraction=1.0,               # Fraction of dataset to use (1.0 = full)

    # ── Hardware ──────────────────────────────────────────
    device=0,                   # 0=GPU
    workers=8,
    amp=True,                   # Automatic Mixed Precision
    cache=False,                # False, "ram", or "disk"

    # ── Model ─────────────────────────────────────────────
    pretrained=True,
    freeze=None,                # Freeze first N layers (int or list of indices)
    single_cls=False,           # Treat all classes as one
    classes=None,               # Train on specific class IDs e.g. [0, 1, 2]

    # ── Optimizer ─────────────────────────────────────────
    optimizer="MuSGD",          # SGD, MuSGD (new in YOLO26), Adam, AdamW, auto
    lr0=0.01,                   # Initial learning rate
    lrf=0.01,                   # Final LR = lr0 * lrf
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    cos_lr=False,               # Use cosine LR scheduler
    nbs=64,                     # Nominal batch size for loss normalization

    # ── Loss Weights ──────────────────────────────────────
    box=7.5,
    cls=0.5,
    cls_pw=0.0,                 # Class weight power (0=disabled, 1=full inverse freq) (Might change to 1.0 as the dataset is not balanced)
    dfl=1.5,                    # NOTE: YOLO26 removes DFL — may have no effect
    pose=12.0,                  # Pose loss (pose models only)         (human body joints)
    kobj=1.0,                   # Keypoint objectness (pose models only)      (humna bosy joints as well)    not out concern

    # ── Augmentation ──────────────────────────────────────
    hsv_h=0.015,
    hsv_s=0.0,        # disabled some since our data is already augmented
    hsv_v=0.0,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    rect=False,
    multi_scale=0.0,            # Random imgsz variation per batch (e.g. 0.25)
    close_mosaic=10,            # Disable mosaic in last N epochs

    # ── Saving & Output ───────────────────────────────────
    project="/content/drive/MyDrive/runs",  # Save directly to GDrive
    name="yolo26s_traffic_signs",
    exist_ok=False,
    save=True,
    save_period=-1,             # Save checkpoint every N epochs (-1=disabled)
    plots=True,

    # ── Validation ────────────────────────────────────────
    val=True,     # run validation on the validation set at the end of each epoch.
    max_det=300,     # Maximum number of detections per image during inference/validation

    # ── Reproducibility ───────────────────────────────────
    seed=42,
    deterministic=True,
    verbose=True,            # detailed logs during training

    # ── Advanced ──────────────────────────────────────────
    compile=False,              # torch.compile (True, False, or "reduce-overhead")
    profile=False,
    resume=False,               # Resume from last checkpoint , we will use itt incase if the runtime disconects
    overlap_mask=True,
    mask_ratio=4,
    dropout=0.0, # remove comma
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Final_Ready_Dataset_RV/Final_Ready_Dataset_RV/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26s_traffic_signs-2, nbs=64, nms=False, opset=None, optimize=False

In [ ]:
# note, epoch 27 and 28 completed, and the logs are stored in results.csv from session one